# Epsilon Fund — CPCV Validation (BB Breakout)

Combinatorial Purged Cross-Validation via `infrastructure/walkforward/cpcv_engine.py`.

Run **after** walk-forward validation to get an unbiased distribution of strategy performance
across all possible train/test splits — not just one rolling path.

---
### What CPCV gives you

| Output | Meaning |
|--------|--------|
| **Path distribution** | 105 complete equity paths (N=8, k=2) — each covers the full dataset once |
| **Mean / Std Sharpe** | Is the strategy reliably positive, or just occasionally lucky? |
| **Parameter stability** | Does the optimizer converge to similar values across all 28 splits? |
| **Tercile comparison** | Do parameter choices in top-performing splits differ from poor ones? |
| **Split heatmap** | Which time periods are structurally hard for this strategy? |

---
### Workflow

1. Complete walk-forward validation for this asset first.
2. Copy `strategy_fn`, `PARAM_DEFS`, and `FIXED_PARAMS` from the walk-forward notebook into the cells below.
3. Set `WF_SHARPE` to the combined OOS Sharpe from walk-forward (for comparison annotation).
4. Run all cells top-to-bottom.
5. Save the results pickle for portfolio-level analysis.

---
### Interpreting results

| Signal | Meaning | Action |
|--------|---------|-------|
| Mean path Sharpe ≈ WF Sharpe, low std | Consistent edge across all splits | High confidence — proceed |
| Mean path Sharpe positive but high std | Strategy works but period-sensitive | Review split heatmap for weak regimes |
| Many negative-Sharpe paths | Edge is concentrated in lucky folds | Revisit strategy architecture |
| CV < 0.10 across most free params | Parameters are stable, not noise-fitted | Safe to fix more params and re-run WF |
| High tercile separation on a param | That param drives performance | Consider narrowing its range or fixing it |

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # <- Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))


from binance_client import get_binance_client, get_data
from cpcv_engine import (
    run_cpcv,
    cpcv_parameter_analysis,
    cpcv_print_param_suggestions,
    cpcv_summary,
    cpcv_confidence_intervals,
    cpcv_ci_summary,
)
from cpcv_visualizer import (
    plot_cpcv_results,
    plot_parameter_distributions,
    plot_parameter_correlation_matrix,
    plot_split_performance_heatmap,
    plot_tercile_comparison,
)

---
## Data

**Use the same pair, interval, and lookback as the corresponding walk-forward notebook**
so the dataset is identical.

All BB breakout walk-forward notebooks use `INTERVAL = '1h'` and `LOOKBACK = 2151` days.
Do not change these when copying to a per-asset notebook — only change `SYMBOL`.

In [ ]:
SYMBOL   = 'BTCUSDT'  # <- change per asset
INTERVAL = '1h'       # <- all BB breakout assets use 1h; do not change
LOOKBACK = 2151       # days — matches walk-forward notebooks exactly

# ── CPCV configuration ────────────────────────────────────────────────────────
N          = 8      # groups to partition data into  ->  C(8,2) = 28 splits
K          = 2      # groups held out per split
N_TRIALS   = 400    # Optuna trials per split (match walk-forward N_TRIALS)
BURNIN     = 200    # bars prepended before each test group for indicator warmup
            # 200 bars = walk-forward BURNIN_BARS — covers max bb_period x4 on resampled 4H
COST       = 0.001  # per-leg trading cost (match walk-forward COST)
PURGE_BARS = 1      # training bars removed at each train/test boundary

# Bar count context:
# LOOKBACK=2151 days at 1h bars -> ~51,600 bars total
# N=8 groups -> ~6,450 bars/group -> ~269 days (~9 months) per group
# Each CPCV path stitches N/K=4 test groups -> covers all ~51,600 bars once

# ── walk-forward Sharpe for comparison annotation (optional) ──────────────────
WF_SHARPE  = None   # <- set to combined OOS Sharpe from walk-forward notebook


client = get_binance_client()
df     = get_data(client, SYMBOL, INTERVAL, LOOKBACK)
print(f'Data: {df.index[0].date()} -> {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

---
## Strategy, PARAM_DEFS, and FIXED_PARAMS

> **Paste `strategy_fn`, `PARAM_DEFS`, and `FIXED_PARAMS` from the corresponding
> walk-forward notebook (`bb_breakout_wf/{ASSET}.ipynb`). These must match exactly —
> same function, same fixing decisions, same search ranges.**

The CPCV engine passes the same `(df_slice, params)` interface as the walk-forward engine.
`strategy_fn` must return `(strategy_df, indicator_cols)`.  
Do not change the function signature or indicator columns.

**Important:** The BB breakout strategy resamples 1H data to 4H internally — this is
handled inside `my_strategy` and requires no changes here. The BURNIN=200 bars
ensures the 4H resampling warmup is satisfied (200 1H bars = 50 4H bars) before any
test group begins.

In [ ]:
# ── parameter search space ────────────────────────────────────────────────────
# Paste from walk-forward notebook — must be identical.
PARAM_DEFS = {
    # ── paste PARAM_DEFS here ──
}

In [ ]:
# ── anchored params ───────────────────────────────────────────────────────────
# Paste from walk-forward notebook — must be identical.
FIXED_PARAMS = {
    # ── paste FIXED_PARAMS here ──
}

In [ ]:
# ── strategy function ─────────────────────────────────────────────────────────
# Paste from walk-forward notebook — must be identical.
# Required signature: my_strategy(df_slice: pd.DataFrame, params: dict)
# Required return:    (strategy_df, indicator_cols)

def my_strategy(df_slice: pd.DataFrame, params: dict):
    raise NotImplementedError(
        'Paste strategy_fn from the corresponding walk-forward notebook'
    )

---
## Score and Reject Functions

Paste the `score_fn` and `reject_fn` from the corresponding walk-forward notebook,
or keep the defaults below. The CPCV engine uses the same scoring interface as the
walk-forward engine.

**Note on `reject_fn` thresholds:** Each CPCV test group covers ~269 days (~9 months).
Trade counts per group will be lower than a full WF fold. The default below uses
`num_trades < 5` instead of the WF notebook's `< 15` to avoid over-rejecting sparse
but valid periods.

In [ ]:
# ── scoring function ─────────────────────────────────────────────────────────
# Paste from the walk-forward notebook, or keep this default.
# Set SCORE_FN = None to use the engine default.

def SCORE_FN(metrics):
    SHARPE_MAX = 3.5
    CALMAR_MAX = 16.0
    RETURN_MAX = 100.0
    calmar = (metrics['total_return'] / abs(metrics['max_drawdown'])
              if metrics['max_drawdown'] != 0 else 0.0)
    s = np.clip(metrics['sharpe_ratio'] / SHARPE_MAX, 0, 1)
    c = np.clip(calmar                  / CALMAR_MAX, 0, 1)
    r = np.clip(metrics['total_return'] / RETURN_MAX, 0, 1)
    return 0.50 * s + 0.30 * c + 0.20 * r

# ── rejection filter ─────────────────────────────────────────────────────────
# Trials that return True are scored -999 and discarded by Optuna.
# Set REJECT_FN = None to use the engine default.
# Thresholds are looser than the WF notebook because CPCV groups are ~9 months,
# not the full 2-year IS window — fewer trades per group is expected.

def REJECT_FN(metrics):
    if metrics is None:                return True
    if metrics['num_trades']   < 5:    return True
    if metrics['win_rate']     < 0.35: return True
    if metrics['max_drawdown'] < -0.80: return True
    if metrics['profit_factor'] < 0.5: return True
    return False

---
## Run CPCV

Partitions the data into `N` equal groups and optimises on every `C(N, K)` training
complement. Each of the 28 splits produces an OOS evaluation on the `K` held-out groups.
All groups are then stitched into 105 complete equity paths (for N=8, k=2).

| Parameter | Description | Value |
|-----------|-------------|-------|
| `N` | Groups — 8 gives C(8,2)=28 splits and 105 complete paths | `8` |
| `K` | Groups held out per split | `2` |
| `N_TRIALS` | Optuna trials per split — match walk-forward for consistency | `400` |
| `BURNIN` | Bars prepended to each test group for indicator warmup | `200` |
| `PURGE_BARS` | Training bars purged at each train/test boundary | `1` |
| `COST` | Per-leg trading cost fraction | `0.001` |

> **Runtime:** `C(N, K) × N_TRIALS` Optuna evaluations.  
> For N=8, K=2, N_TRIALS=400: 28 × 400 = 11,200 backtests.  
> 1H data is heavier per backtest than daily — expect 30–90 min depending on hardware.

In [ ]:
results = run_cpcv(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    N            = N,
    k            = K,
    purge_bars   = PURGE_BARS,
    n_trials     = N_TRIALS,
    burnin       = BURNIN,
    cost         = COST,
    score_fn     = SCORE_FN,
    reject_fn    = REJECT_FN,
    verbose      = True,
)

### CPCV Summary — display flags

`cpcv_summary` is split into two cells so the output stays readable. Each flag controls one section of the printed report. Toggle them to focus on what you need.

| Flag | Default | What it prints |
|---|---|---|
| `show_group_legend` | `True` | **Group date ranges** — maps each group number (g1–g8) to its calendar window. |
| `show_distribution` | `True` | **Path distribution table** — Mean / Std / Min / Q25 / Median / Q75 / Max / IQR across all 105 paths. |
| `show_paths` | `False` | **Full per-path table** — one row per path (105 rows). |
| `show_highlights` | `True` | **Top 5 / Bottom 5 paths** — ranked by Sharpe. |
| `show_split_legend` | `False` | **Split legend** — decodes the `gN->sM` notation. |
| `show_ci` | `True` | **Confidence intervals** — overlap-adjusted CI for Sharpe, Calmar, and Max DD. |

In [ ]:
# Group legend + path distribution percentile table
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = True,
    show_paths        = False,
    show_highlights   = False,
    show_split_legend = False,
    show_ci           = True,
)

In [ ]:
# Top/bottom highlights + split legend
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = False,
    show_paths        = False,
    show_highlights   = True,
    show_split_legend = True,
    show_ci           = False,
)

---
## Path-Level Charts

Three outputs:
- **Equity curves** — all 105 paths (semi-transparent blue), the mean path (bold amber),
  and the min-max envelope. Vertical dotted lines mark group boundaries g0-g7.
- **Sharpe histogram** — distribution of the 105 path-level Sharpe ratios with an
  overlap-adjusted confidence interval overlay.
- **Confidence interval text table** — naive and overlap-adjusted 95% CIs for Sharpe,
  Calmar, and Max DD.

In [ ]:
ci = cpcv_confidence_intervals(results)
plot_cpcv_results(results, wf_sharpe=WF_SHARPE, ci_results=ci);

---
## Parameter Analysis

Compute the analysis object once — all four charts below read from it.

In [ ]:
analysis = cpcv_parameter_analysis(results)

# Prints consensus_ranges table + copy-pasteable PARAM_DEFS / FIXED_PARAMS blocks.
# CV < 0.10  -> "fix at {median}"   : converges consistently, safe to anchor.
# CV 0.10-0.25 -> "narrow to IQR"  : reduce search range to Q25-Q75 next WF run.
# CV > 0.25  -> "keep current range": period-sensitive, keep free.
cpcv_print_param_suggestions(results, analysis)

### Parameter Distributions

One subplot per free parameter. Each dot is one split's best value, jittered horizontally
and **coloured by that split's OOS Sharpe** (red = low, green = high).

- **IQR band** (shaded blue) — the middle 50% of optimised values across splits.
- **Median line** (amber dashed) — the central tendency.
- **Subplot subtitle** — CV verdict (Stable / Moderate / Scattered) and recommended action.

In [ ]:
plot_parameter_distributions(results, analysis=analysis);

### Parameter Correlation Matrix

Pearson correlation between every pair of free parameters across the 28 splits.

- **Red** (r close to -1) — substitutable parameters; consider fixing one.
- **Blue** (r close to +1) — parameters that move together; one may be redundant.
- **White** (r ~= 0) — independent.

Any |r| > 0.6 pair is worth investigating for potential over-parameterisation.

In [ ]:
plot_parameter_correlation_matrix(analysis);

### Split Performance Heatmap

An N x N grid where cell (i, j) shows the **OOS Sharpe of the split that held out
groups i and j** for testing.

- **Green** — that group-pair was OOS-profitable.
- **Red** — those two periods were structurally hard OOS.

A consistent block of red around a particular group index means one time period is
structurally difficult regardless of which other period it's paired with. Cross-reference
with the group date legend in `cpcv_summary` to identify that regime.

In [ ]:
plot_split_performance_heatmap(results);

### Tercile Comparison

One subplot per free parameter showing **top third vs bottom third of splits by OOS Sharpe**.

| Separation | Interpretation |
|------------|----------------|
| < 0.5 | No meaningful difference — parameter choice doesn't predict split quality |
| 0.5-1.0 | Moderate signal — consider narrowing the range toward the top-tercile median |
| > 1.0 | Strong signal — this parameter's value is predictive of OOS performance |

In [ ]:
plot_tercile_comparison(results, analysis);

---
## Save Results

Pickle the full results dict for portfolio-level analysis.

Naming convention: `oos/{SYMBOL.lower()}_{INTERVAL}_cpcv.pkl`  
Example: BTCUSDT at 1h -> `oos/btcusdt_1h_cpcv.pkl`

The portfolio notebook (`portfolio_cpcv.ipynb`) loads these pkl files by asset name.

In [ ]:
os.makedirs('oos', exist_ok=True)
results_file = os.path.join('oos', f'{SYMBOL.lower()}_{INTERVAL}_cpcv.pkl')
pd.to_pickle(results, results_file)
print(f'Saved to {results_file}')